In [1]:
# For debugging
%load_ext autoreload
%autoreload 2

In [2]:
from copy import deepcopy

import torch
import torch.nn
import torch.optim
from torch.utils.data import DataLoader

import xarray as xr
import numpy

from tqdm.notebook import tqdm 

from neural_net import get_net
from constants import *

import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'einops'

In [ ]:
torch.manual_seed(42)

# Settings

In [43]:
device = torch.device("cuda")
dtype = torch.bfloat16

In [ ]:
#batch_size = 512
#n_epochs = 20
#n_blocks = 4
# n_features = 64 

# size of the smaller datasets in which I break down the dataset 
batch_size = 2               # small to avoid OOM; raise if memory allows
# how many loops for training 
n_epochs = 1
# data and 
n_blocks = 2
n_features = 32

# Load data

In [ ]:
ds_train = xr.open_zarr("../data/sqg_train_small.zarr")["q"].compute(num_workers=4) # training data 

# Normalize data
train_data = torch.cat((
    (torch.as_tensor(ds_train.values[:-1], dtype=dtype)-in_mean) / in_std,
    (torch.as_tensor(ds_train.values[1:]-ds_train.values[:-1], dtype=dtype)-res_mean) / res_std
), dim=1)

del ds_train

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
print("len train loader:", len(train_loader))

In [46]:
ds_val = xr.open_zarr("../data/sqg_val.zarr")["q"].compute(num_workers=16) # validation data 
val_data = torch.cat((
    (torch.as_tensor(ds_val.values[:-1], dtype=dtype)-in_mean) / in_std,
    (torch.as_tensor(ds_val.values[1:]-ds_val.values[:-1], dtype=dtype)-res_mean) / res_std
), dim=1)
del ds_val

val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Define neural network

Use of a convolutional neural network

In [ ]:
# CHANGE THIS IF NOT CONDITIONED 

convnet = get_net(
    # Input: intermediate state (1) + initial conditions (1)
    n_input=2,
    n_output=1, n_blocks=n_blocks, n_features=n_features, mult=2,
    # Activation of pseudo time
    n_embedding=256, wave_length=0.1,
    device=device, dtype=dtype
)
optim = torch.optim.Adam(convnet.parameters(), lr=1E-3)

In [ ]:
pbar_epoch = tqdm(range(n_epochs))

mse_val = torch.inf
best_mse = torch.inf
best_model = None

for _ in pbar_epoch:
    # length of this = len training dataset
    pbar_train = tqdm(iter(train_loader), total=len(train_loader), leave=True)
    
    # Training loop
    convnet = convnet.train()
    # loop over the whole training dataset 
    for batch in pbar_train:        
        batch = batch.to(device=device, dtype=dtype)
        
        # split each batch into two parts:inout/conditioning and data we want to generate 
        # My batch has shape [B, 2, H, W] and gets split into two [B, 1, H, W] tensors
        data_in, data_target = batch.split((1, 1), dim=1)

        ## Sample epsilon (z_0) as random draw from a normal distribution
        noise = torch.randn_like(data_target)

        ## Stratified sampling of pseudo time to reduce variance during optimisation (Kingma et al., 2022)
        # ensures more uniform coverage of the time space and reduces variance during training 
        ## Take pseudo time between [0, 1)
        pseudo_time = torch.linspace(0, 1, batch.shape[0]+1, device=device, dtype=dtype)[:-1, None]
        ### Introduce random shift between 0 and 1
        time_shift = torch.rand(1, device=device, dtype=dtype)
        pseudo_time = pseudo_time + time_shift
        ### Ensure that pseudo time is between 0 and 1
        pseudo_time = pseudo_time%1

        ## Construct intermediate state (z_t) with linear interpolant
        intermediate_state = pseudo_time[..., None, None] * data_target \
            + (1-pseudo_time[..., None, None]) * noise
        target_velocity = data_target - noise # v = x1 - x0 

        ## Neural network input = [intermediate state, initial conditions]
        ## Conditional flow matching: input = noisy intermediate state at time t and the conditioning information (initial conditions)

        input_tensor = torch.cat((
            intermediate_state, data_in
            
        ), dim=1)                                 

        optim.zero_grad()
        ## Neural network predicts velocity now
        prediction = convnet(input_tensor, pseudo_time)
        # Downsample target to 64x64 to match model output resolution
        # does it have a bad consequence? is it better to upsample the net output instead? or just train full resolution? 
        target_velocity_downsampled = torch.nn.functional.avg_pool2d(
            target_velocity, kernel_size=, stride=8
        )

        error = (prediction - target_velocity_downsampled).pow(2)

        mse_train = (error).mean()
        mse_train.backward()
        optim.step()

        # increment bar 
        pbar_train.set_postfix(mse_train=mse_train.item(), mse_val=mse_val)

    mse_val = 0
    samples_val = 0
    pbar_val = tqdm(enumerate(val_loader), total=len(val_loader), leave=False)
    
    # Validation loop
    convnet = convnet.eval()
    for k, batch in pbar_val:        
        batch = batch.to(device=device, dtype=dtype)
        data_in, data_target = batch.split((1, 1), dim=1)

        ## Sample epsilon (z_0) as random draw from a normal distribution
        noise = torch.randn_like(data_target)

        ### Take pseudo time between [0, 1)
        pseudo_time = torch.linspace(0, 1, batch.shape[0]+1, device=device, dtype=dtype)[:-1, None]
        time_shift = torch.rand(1, device=device, dtype=dtype)
        pseudo_time = pseudo_time + time_shift
        pseudo_time = pseudo_time%1

        ## Construct intermediate state (z_t) with linear interpolant
        intermediate_state = pseudo_time[..., None, None] * data_target \
            + (1-pseudo_time[..., None, None]) * noise
        target_velocity = data_target-noise

        ## Neural network input = [intermediate state, initial conditions]
        input_tensor = torch.cat((
            intermediate_state, data_in
        ), dim=1)         
        
        with torch.no_grad():
            prediction = convnet(input_tensor, pseudo_time)
            # Downsample target to 64x64 to match model output resolution
            target_velocity_downsampled = torch.nn.functional.avg_pool2d(
                target_velocity, kernel_size=8, stride=8
            )
        error = (prediction - target_velocity_downsampled).pow(2)
        curr_se = (error).mean(dim=(1, 2, 3)).sum().item()
        mse_val = mse_val * samples_val + curr_se
        samples_val = samples_val + len(batch)
        mse_val = mse_val / samples_val

    pbar_train.set_postfix(mse_train=mse_train.item(), mse_val=mse_val)
        
    # Check if new model is better
    if mse_val < 0.999 * best_mse: # 0.999 to get rid of randomness 
        # the best model becomes the one with smaller validation MSE 
        best_mse = mse_val
        best_model = deepcopy(convnet).cpu()
    # best model is saved, no early stopping 

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

# Store best model

In [49]:
state_dict = best_model.cpu().state_dict()
torch.save(state_dict, "../data/test_flow_model.ckpt")